# Practical Statistics for Data Scientists (Python)
# Chapter 2 — Data and Sampling Distributions

> **Author:** Satrio Arjuna Putra  
> **Reference:** *Practical Statistics for Data Scientists* — Bruce, Bruce & Gedeck (O'Reilly)

---

## 📋 Chapter Summary

Chapter 2 focuses on **sampling** and **statistical distributions** — fundamental to understanding uncertainty in data science.

| Topic | Key Concepts |
|-------|-------------|
| **Random Sampling** | Population vs sample, sampling bias, random seed |
| **Sampling Distribution** | Distribution of a statistic across many samples |
| **Bootstrap** | Resampling with replacement to estimate uncertainty |
| **Confidence Intervals** | Range of plausible values for a parameter |
| **Normal Distribution** | Bell curve, z-scores, standard normal |
| **Long-Tailed Distributions** | Skewness, heavy tails (power laws) |
| **t-Distribution** | Use when sample size is small |
| **Binomial Distribution** | Count of successes in n trials |
| **Poisson Distribution** | Count of rare events in a fixed interval |

---


## 🔧 Setup & Imports

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample

DATA_BASE = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"
LOANS_INCOME = DATA_BASE + "loans_income.csv"

np.random.seed(42)
print("Libraries loaded!")


---

## 1. Random Sampling and Sample Bias

### 📚 Theory

A **population** is the complete set of items of interest. A **sample** is a subset.

**Random sampling** ensures each item has an equal chance of selection, minimising **bias**.

**Types of bias:**
- **Selection bias:** Systematic exclusion of certain groups
- **Survivorship bias:** Only looking at "survivors" (e.g., successful companies)
- **Confirmation bias:** Cherry-picking data that supports your hypothesis

> **Key Insight:** Even large samples can be biased if the sampling method is flawed. Random sampling guards against bias.


In [ ]:
# Load loans income data
loans_income = pd.read_csv(LOANS_INCOME, squeeze=True)
print("Population size:", len(loans_income))
print("Population mean:", loans_income.mean())
print("Population median:", loans_income.median())
print(loans_income.describe())


In [ ]:
# Draw a simple random sample
sample_data = pd.DataFrame({
    'income': loans_income.sample(1000, random_state=42),
    'type': 'Data'
})
print("Sample mean:", sample_data['income'].mean())
print("Sample median:", sample_data['income'].median())


---

## 2. Sampling Distribution of a Statistic

### 📚 Theory

The **sampling distribution** is the distribution of a statistic (e.g., mean) computed from many different samples drawn from the same population.

**Central Limit Theorem (CLT):**
> The sampling distribution of the sample mean approaches a **normal distribution** as sample size increases, regardless of the population distribution.

$$\bar{X} \sim N\left(\mu, \frac{\sigma^2}{n}\right)$$

**Standard Error:** $SE = \frac{\sigma}{\sqrt{n}}$ — measures how much the sample mean varies.

> **Key Insight:** CLT allows us to make probability statements about sample means even when the underlying distribution is not normal.


In [ ]:
# Demonstrate the CLT: simulate sampling distribution of the mean
results = []
for _ in range(1000):
    sample = loans_income.sample(n=1000, replace=True)
    results.append(sample.mean())

sampling_dist = pd.Series(results)
print("Mean of sampling distribution:", sampling_dist.mean())
print("Std of sampling distribution (Standard Error):", sampling_dist.std())
print("Theoretical SE:", loans_income.std() / np.sqrt(1000))


In [ ]:
# Plot the sampling distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

loans_income.plot.hist(ax=axes[0], bins=40)
axes[0].set_title('Original Population Distribution')
axes[0].set_xlabel('Income ($)')

sampling_dist.plot.hist(ax=axes[1], bins=40)
axes[1].set_title('Sampling Distribution of the Mean (n=1000)')
axes[1].set_xlabel('Sample Mean Income ($)')

plt.tight_layout()
plt.show()


---

## 3. The Bootstrap

### 📚 Theory

The **bootstrap** is a resampling technique that estimates the sampling distribution of any statistic *without* mathematical formulas.

**Algorithm:**
1. Draw n samples *with replacement* from your data (a "bootstrap sample")
2. Compute the statistic of interest
3. Repeat B times (e.g., 1000)
4. The distribution of B statistics = bootstrap distribution

> **Key Insight:** The bootstrap is model-free and works for almost any statistic (mean, median, correlation, etc.).


In [ ]:
# Bootstrap example: 95% confidence interval for the median
sample = loans_income.sample(n=1000, random_state=42)

bootstrap_medians = []
for _ in range(1000):
    boot_sample = sample.sample(n=len(sample), replace=True)
    bootstrap_medians.append(boot_sample.median())

bootstrap_medians = pd.Series(bootstrap_medians)
ci_low, ci_high = bootstrap_medians.quantile(0.025), bootstrap_medians.quantile(0.975)

print(f"Sample Median: {sample.median():,.0f}")
print(f"Bootstrap 95% CI: ({ci_low:,.0f}, {ci_high:,.0f})")

bootstrap_medians.plot.hist(bins=30, figsize=(8, 4))
plt.axvline(ci_low, color='red', linestyle='--', label=f'2.5% = {ci_low:,.0f}')
plt.axvline(ci_high, color='red', linestyle='--', label=f'97.5% = {ci_high:,.0f}')
plt.xlabel('Median Income')
plt.title('Bootstrap Distribution of Median')
plt.legend()
plt.tight_layout()
plt.show()


---

## 4. Confidence Intervals

### 📚 Theory

A **confidence interval (CI)** gives a range of plausible values for a population parameter.

**Interpretation:** A 95% CI means: if we repeated the sampling procedure many times, 95% of the CIs would contain the true parameter.

$$\bar{x} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}$$

| Level | z-value |
|-------|---------|
| 90% | 1.645 |
| 95% | 1.960 |
| 99% | 2.576 |

> **Key Insight:** A wider CI = more uncertainty (smaller sample or higher variance). Confidence ≠ probability the true value is in the interval.


In [ ]:
# 95% CI for the mean using CLT
sample = loans_income.sample(n=1000, random_state=42)
n = len(sample)
mean = sample.mean()
se = sample.std() / np.sqrt(n)
z = 1.96  # 95% confidence

ci_low = mean - z * se
ci_high = mean + z * se
print(f"Sample Mean: {mean:,.0f}")
print(f"95% CI (CLT): ({ci_low:,.0f}, {ci_high:,.0f})")


---

## 5. Normal Distribution

### 📚 Theory

The **normal distribution** (bell curve) is the most important distribution in statistics.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

**Standard Normal:** $Z = \frac{X - \mu}{\sigma} \sim N(0,1)$

**68-95-99.7 Rule:**
- 68% of data within ±1σ
- 95% within ±2σ  
- 99.7% within ±3σ

**Q-Q Plot:** Compares the quantiles of your data to a theoretical normal distribution. Points on the diagonal line = normal data.

> **Key Insight:** Many statistical tests assume normality. Always check with a histogram or Q-Q plot.


In [ ]:
# Normal distribution visualisation
x = np.linspace(-4, 4, 200)
pdf = stats.norm.pdf(x)
plt.figure(figsize=(8, 4))
plt.plot(x, pdf, 'b-', linewidth=2)
plt.fill_between(x, pdf, where=(x >= -1) & (x <= 1), alpha=0.3, label='68% (±1σ)')
plt.fill_between(x, pdf, where=((x >= -2) & (x < -1)) | ((x > 1) & (x <= 2)),
                 alpha=0.2, color='orange', label='95% (±2σ)')
plt.xlabel('Standard Deviations from Mean')
plt.ylabel('Density')
plt.title('Standard Normal Distribution')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Q-Q Plot to check normality
sample = loans_income.sample(n=200, random_state=42)
fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(sample, dist='norm', plot=ax)
ax.set_title('Q-Q Plot: Loan Income Sample')
plt.tight_layout()
plt.show()


---

## 6. Long-Tailed Distributions

### 📚 Theory

A **long-tailed** (heavy-tailed) distribution has much more probability in the tails than a normal distribution.

Examples: **Income**, **website traffic**, **earthquake magnitudes** (power law distributions).

**Consequence:** Means and standard deviations can be misleading. Extreme values are much more common than the normal model predicts.

> **Key Insight:** Financial returns, social networks, and natural disasters often follow heavy-tailed distributions — normal models underestimate extreme event probability.


In [ ]:
# Visualise income distribution: long tail
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

loans_income.plot.hist(bins=50, ax=axes[0])
axes[0].set_xlabel('Income')
axes[0].set_title('Income Distribution (raw)')

np.log1p(loans_income).plot.hist(bins=50, ax=axes[1])
axes[1].set_xlabel('Log Income')
axes[1].set_title('Log-Transformed Income Distribution')

plt.tight_layout()
plt.show()


---

## 7. t-Distribution

### 📚 Theory

The **t-distribution** is used instead of the normal when:
- Sample size is small (n < 30), OR
- Population standard deviation (σ) is unknown

$$t = \frac{\bar{x} - \mu}{s / \sqrt{n}}$$

As n → ∞, t-distribution → Normal distribution.

**Degrees of freedom (df):** df = n - 1. More df = closer to normal.

> **Key Insight:** Always use t-distribution for small samples when σ is unknown. For large samples, the difference from normal is negligible.


In [ ]:
# Compare t-distribution to normal for different sample sizes
x = np.linspace(-4, 4, 200)
plt.figure(figsize=(8, 4))
plt.plot(x, stats.norm.pdf(x), 'k-', label='Normal', linewidth=2)
for df in [2, 5, 10, 30]:
    plt.plot(x, stats.t.pdf(x, df), '--', label=f't (df={df})')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('t-Distribution vs Normal Distribution')
plt.legend()
plt.tight_layout()
plt.show()


---

## 8. Binomial and Poisson Distributions

### 📚 Theory

**Binomial Distribution:** Number of successes in n independent trials, each with probability p.
$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**Poisson Distribution:** Number of rare events in a fixed interval, given average rate λ.
$$P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

> **Key Insight:** Binomial → Poisson as n→∞ and p→0, keeping np=λ constant. Poisson is used for modelling counts of rare events (calls per hour, defects per unit).


In [ ]:
# Binomial: simulate 200 coin flips (p=0.5), 1000 times
binom_results = [np.random.binomial(200, 0.5) for _ in range(1000)]
pd.Series(binom_results).plot.hist(bins=30, figsize=(8, 4))
plt.xlabel('Number of Heads in 200 Flips')
plt.title('Binomial Distribution (n=200, p=0.5)')
plt.tight_layout()
plt.show()


In [ ]:
# Poisson: average 2 events per hour
poisson_results = np.random.poisson(lam=2, size=1000)
pd.Series(poisson_results).plot.hist(bins=15, figsize=(8, 4), rwidth=0.8)
plt.xlabel('Number of Events')
plt.title('Poisson Distribution (λ=2)')
plt.tight_layout()
plt.show()


---

## ✅ Chapter 2 Key Takeaways

1. **Random sampling** guards against bias; large biased samples are worse than small random ones.
2. **CLT:** Sample means are approximately normally distributed for large n — regardless of population shape.
3. **Bootstrap:** Model-free way to estimate uncertainty for *any* statistic.
4. **Confidence Intervals:** Quantify uncertainty; wider = more uncertainty.
5. **Normal distribution:** Fundamental to statistics, but many real-world datasets are skewed or heavy-tailed.
6. **t-distribution:** Use for small samples or when σ is unknown.
7. **Binomial/Poisson:** Model counts of events with known probabilities or rates.

---
*Reference: Chapter 2, Practical Statistics for Data Scientists (O'Reilly)*
